In [ ]:
# -*- coding: utf-8 -*-
import ee
import os
import sys
import json
import folium

In [ ]:
# Authentication: run `earthengine authenticate` once and set the environment
# variable EE_PROJECT to your own Google Cloud project before executing.
import os
import ee
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project=os.environ.get('EE_PROJECT'))
#ee.Initialize(project=os.environ.get('EE_PROJECT'))
import geemap.core as geemap
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()

In [ ]:
# =================== PARÂMETROS ===================
START_DATE   = '1985-01-01'
END_DATE     = '2024-12-01'

ASSET_CITY   = 'projects/ee-rnvuefsppgm/assets/CAP_JOIN2'
ASSET_PREFIX = 'projects/rnvuefsppgm/assets/RSEI_CITIES2'   # pasta de destino no Asset

# >>>>>> DEFINA AQUI os municípios a processar (CD_MUN) <<<<<<
# Ex.: ['2927408']  ou  ['2927408','2927407']
CODMUN_FILTER = ['2927408']


In [ ]:
# =================== DADOS GEOMÉTRICOS ===================
# Carrega o asset e adiciona marcador binário (id_cod) para usos futuros
cities_fc_full = ee.FeatureCollection(ASSET_CITY).map(lambda f: f.set('id_cod', 1))

# Filtra APENAS as feições desejadas (server-side)
if not isinstance(CODMUN_FILTER, list) or len(CODMUN_FILTER) == 0:
    raise ValueError("Defina CODMUN_FILTER como lista de CD_MUN, ex.: ['2927408'].")

cities_fc = cities_fc_full.filter(ee.Filter.inList('CD_MUN', CODMUN_FILTER))

print('Total no asset completo:', cities_fc_full.size().getInfo())
print('Total selecionado (CODMUN_FILTER):', cities_fc.size().getInfo())

In [ ]:
# =================== MÁSCARA E HARMONIZAÇÃO (C2 L2) ===================
def maskLandsatC2L2(img):
    """Máscara de nuvem/sombra/neve (QA_PIXEL) + saturação (QA_RADSAT)."""
    qa = img.select('QA_PIXEL')
    notDilated = qa.bitwiseAnd(1 << 1).eq(0)
    notCloud   = qa.bitwiseAnd(1 << 3).eq(0)
    notShadow  = qa.bitwiseAnd(1 << 4).eq(0)
    notSnow    = qa.bitwiseAnd(1 << 5).eq(0)
    isClear    = qa.bitwiseAnd(1 << 6).neq(0)

    hasRadSat = img.bandNames().contains('QA_RADSAT')
    notSaturated = ee.Image(ee.Algorithms.If(
        hasRadSat, img.select('QA_RADSAT').eq(0), ee.Image(1)
    ))
    mask = ee.Image(notDilated).And(notCloud).And(notShadow).And(notSnow).And(isClear).And(notSaturated)
    return img.updateMask(mask)

In [ ]:
def harmonizeLandsatC2L2_L89(img):
    """L8/L9: SR→[blue..swir2], ST→'st' (Kelvin); adiciona índices RSEI."""
    img = maskLandsatC2L2(img)

    # Ópticas (escala 0.0000275, offset -0.2)
    oliSR = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
    outSR = ['blue','green','red','nir','swir1','swir2']
    srSelected = img.select(oliSR).multiply(0.0000275).add(-0.2).rename(outSR)

    # ST (Kelvin): ST_B10 * 0.00341802 + 149.0
    stKelvin = img.select('ST_B10').multiply(0.00341802).add(149.0).rename('st')

    out = srSelected.addBands(stKelvin).toFloat()
    out = addIndices(out)  # RSEI components
    return out.copyProperties(img, img.propertyNames())

In [ ]:
def harmonizeLandsatC2L2_L57(img):
    """L5/L7: SR→[blue..swir2], ST→'st' (Kelvin); adiciona índices RSEI."""
    img = maskLandsatC2L2(img)

    tmSR  = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
    outSR = ['blue','green','red','nir','swir1','swir2']
    srSelected = img.select(tmSR).multiply(0.0000275).add(-0.2).rename(outSR)

    # ST (Kelvin): ST_B6 * 0.00341802 + 149.0
    stKelvin = img.select('ST_B6').multiply(0.00341802).add(149.0).rename('st')

    out = srSelected.addBands(stKelvin).toFloat()
    out = addIndices(out)  # RSEI components
    return out.copyProperties(img, img.propertyNames())

In [ ]:
# =================== ÍNDICES (RSEI: NDBSI + NDVI + LST + Wetness) ===================
def addIndices(img):
    """
    Adiciona (para RSEI - Xu, 2008):
      - NDBSI = (IBI + BSI)/2  [IBI com NDVI e MNDWI; SI = BSI]
      - NDVI
      - LST (°C) a partir de 'st' (ou ST_B10/ST_B6, se existirem)
      - Tasseled Cap (TCB, TCG, TCW) e 'wetness' (TCW)
    Requer: blue, green, red, nir, swir1, swir2 (e preferencialmente 'st').
    """
    eps = ee.Image.constant(1e-6)

    # Base mask de banda única (evita conflito de dimensões ao final)
    baseMask = img.select(0).mask()

    # -- Dryness (NDBSI) --
    ndvi  = img.normalizedDifference(['nir',   'red']).rename('NDVI')
    mndwi = img.normalizedDifference(['green', 'swir1']).rename('MNDWI')
    ndbi  = img.normalizedDifference(['swir1', 'nir']).rename('NDBI')

    sum_sr = img.select(['swir1', 'red']).reduce(ee.Reducer.sum())   # SWIR1 + RED
    sum_nb = img.select(['nir',  'blue']).reduce(ee.Reducer.sum())   # NIR  + BLUE
    bsi = sum_sr.subtract(sum_nb).divide(sum_sr.add(sum_nb).add(eps)).rename('BSI')

    comp = ndvi.add(mndwi).divide(2.0)
    ibi  = ndbi.subtract(comp).divide(ndbi.add(comp).add(eps)).rename('IBI')

    ndbsi = ibi.add(bsi).divide(2.0).rename('NDBSI')

    # -- Heat (LST em °C) --
    bnames = img.bandNames()
    has_st      = bnames.contains('st')
    has_st_b10  = bnames.contains('ST_B10')
    has_st_b6   = bnames.contains('ST_B6')

    lstK = ee.Image(ee.Algorithms.If(
        has_st, img.select('st'),
        ee.Algorithms.If(has_st_b10, img.select('ST_B10'),
                         ee.Algorithms.If(has_st_b6, img.select('ST_B6'), ee.Image(0).selfMask()))
    )).rename('LST_K').toFloat().updateMask(baseMask)
    lstC = lstK.subtract(273.15).rename('LST').toFloat().updateMask(baseMask)

    # -- Wetness (TCW via Tasseled Cap) --
    X = img.select(['blue','green','red','nir','swir1','swir2']).toArray().toArray(1)

    TCT_TM = ee.Array([
        [ 0.3037,  0.2793,  0.4743,  0.5585,  0.5082,  0.1863],  # TCB
        [-0.2848, -0.2435, -0.5436,  0.7243,  0.0840, -0.1800],  # TCG
        [ 0.1509,  0.1973,  0.3279,  0.3406, -0.7112, -0.4572]   # TCW
    ])
    TCT_OLI = ee.Array([
        [ 0.3029,  0.2786,  0.4733,  0.5599,  0.5080,  0.1872],
        [-0.2941, -0.2430, -0.5424,  0.7276,  0.0713, -0.1608],
        [ 0.1511,  0.1973,  0.3283,  0.3407, -0.7117, -0.4559]
    ])
    has_b10 = bnames.contains('ST_B10')
    A_tct = ee.Array(ee.Algorithms.If(has_b10, TCT_OLI, TCT_TM))
    tcap  = ee.Image(A_tct).matrixMultiply(X).arrayProject([0]).arrayFlatten([['TCB','TCG','TCW']]).updateMask(baseMask)
    wet   = tcap.select('TCW').rename('wetness').updateMask(baseMask)

    # Saída com máscara única aplicada
    add_bands = [
        ndbsi.toFloat().updateMask(baseMask),
        ndvi.toFloat().updateMask(baseMask),
        lstC,
        tcap.select(['TCB','TCG','TCW']).toFloat().updateMask(baseMask),
        wet.toFloat().updateMask(baseMask)
    ]
    return img.addBands(add_bands)

In [ ]:
# =================== FECHAMENTO DE BURACOS POR MÉDIA DOS VIZINHOS (SEM RAIO) ===================
def fill_holes_by_neighbor_mean(img, band='LST', max_iters=200):
    """
    Preenche buracos de 'band' usando média 3x3 iterativa, sem especificar raio.
    A média é aplicada repetidamente e propagada para dentro do buraco até fechá-lo.
    - max_iters controla um teto de iterações (200 cobre buracos grandes).
    """
    bimg = img.select([band])

    def _step(i, prev):
        prev = ee.Image(prev)
        # média 3x3; usamos kernel quadrado 3x3 (não passa raio)
        mean3 = prev.focal_mean(kernel=ee.Kernel.square(1))
        # preenche somente onde está mascarado
        filled = prev.unmask(mean3)
        return filled

    filled = ee.Image(ee.List.sequence(1, max_iters).iterate(_step, bimg))
    return img.addBands(filled.rename(band), overwrite=True)

In [ ]:
# =================== CONSTRUÇÃO DE COLEÇÃO POR MUNICÍPIO ===================
def build_collection_for_city(feat, start_date=START_DATE, end_date=END_DATE):
    """Monta coleção multi-sensor (L5/L7/L8/L9) já harmonizada + índices, filtrada pela geometria."""
    geom = ee.Feature(feat).geometry()

    col_l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
                .filterDate(start_date, end_date)
                .filterBounds(geom)
                .map(harmonizeLandsatC2L2_L57))

    col_l7 = (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
                .filterDate(start_date, end_date)
                .filterBounds(geom)
                .map(harmonizeLandsatC2L2_L57))

    col_l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                .filterDate(start_date, end_date)
                .filterBounds(geom)
                .map(harmonizeLandsatC2L2_L89))

    col_l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
                .filterDate(start_date, end_date)
                .filterBounds(geom)
                .map(harmonizeLandsatC2L2_L89))

    col = ee.ImageCollection(col_l5.merge(col_l7).merge(col_l8).merge(col_l9))

    # >>> aplica o fechamento de buracos APENAS na banda LST <<<
    col = col.map(lambda im: fill_holes_by_neighbor_mean(im, band='LST', max_iters=200))

    # # Se quiser preencher outras bandas também:
    # bands_to_fill = ['LST']  # ex.: ['LST','NDBSI','wetness']
    # col = col.map(lambda im: functools.reduce(
    #     lambda acc, b: fill_holes_by_neighbor_mean(acc, b, 200), bands_to_fill, im))

    return col

In [ ]:
# =================== EXPORTAÇÃO ===================
def export_city_scenes_from_feature(feat):
    """
    Exporta cada cena (com índices) para o Asset, recortada pela geometria do município especificado.
    'feat' deve ser uma ee.Feature previamente filtrada do ASSET_CITY.
    """
    codmun = ee.String(feat.get('CD_MUN')).getInfo()
    nm_mun = ee.String(feat.get('NM_MUN')).getInfo()
    geom   = ee.Feature(feat).geometry()  # ee.Geometry (server-side)

    print(f"[{codmun} | {nm_mun}] preparando coleção…")
    col = build_collection_for_city(feat)

    # IDs das cenas
    scene_ids = col.aggregate_array('system:index').getInfo()
    print(f"[{codmun} | {nm_mun}] {len(scene_ids)} cenas para exportar.")

    # Máscara única da cidade (1 banda)
    mask_city = ee.Image().toByte().paint(geom, 1).rename('mask_city').gt(0)

    for sid in scene_ids:
        img = ee.Image(col.filter(ee.Filter.eq('system:index', sid)).first())

        # recorte por geometria (mask_city é 1 banda)
        img_clip = (
            img.updateMask(mask_city)
               .set('NM_MUN', nm_mun)
               .set('CD_MUN', codmun)
        )

        asset_id = f"{ASSET_PREFIX}/{sid}"
        print(f"  • Exportando -> {asset_id}")

        task = ee.batch.Export.image.toAsset(
            image=img_clip,
            description=sid,
            assetId=asset_id,
            region=geom,           # ee.Geometry (server-side)
            scale=30,
            pyramidingPolicy={'.default': 'mean'},
            maxPixels=1e13
        )
        task.start()

In [ ]:
# =================== EXECUÇÃO (apenas feições filtradas) ===================
selected_feats = cities_fc.toList(cities_fc.size())
count = cities_fc.size().getInfo()
print("Feições selecionadas (CODMUN_FILTER):", CODMUN_FILTER, "| total:", count)

for i in range(count):
    feat = ee.Feature(selected_feats.get(i))
    try:
        export_city_scenes_from_feature(feat)
    except Exception as e:
        print(f"[ERRO] feature #{i}: {e}")

# (Opcional) Visualização rápida da extensão do asset
##try:
#    region = cities_fc.geometry()
#    my_map = folium.Map(location=[-15.78, -47.93], zoom_start=5)
#    my_map.add_child(folium.GeoJson(region.getInfo()))
#    display(my_map)
#except Exception:
#    pass
